**MGMT298D: Science and Strategy of AI**

# Week 6: Building a Tiny LLM


We build a small GPT-style language model and train it on Yelp reviews.

By the end, you should be able to see how a model learns to predict the next token from examples, and how its generations change as training progresses.


# 1 Setup

In [ ]:
#@title Import libraries { display-mode: "form" }
import os
import logging
import warnings

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")

import numpy as np
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, HTML

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


---
# 2 Load Yelp Reviews

#### The full Yelp Review Full training split has 650,000 reviews. We use 50,000 for this classroom demo.


In [ ]:
dataset = load_dataset("yelp_review_full", split="train")

FULL_TRAIN_SIZE = len(dataset)
NUM_REVIEWS = 50_000

texts = dataset["text"][:NUM_REVIEWS]

print(f"Full training split: {FULL_TRAIN_SIZE:,} reviews")
print(f"Using for this notebook: {NUM_REVIEWS:,} reviews")

for i in range(5):
    stars = dataset["label"][i] + 1
    preview = texts[i][:100].replace(chr(10), " ")
    print(f"  [{stars}★] {preview}...")


In [ ]:
#@title Tokenize reviews { display-mode: "form" }
VOCAB_SIZE = 10000
SEQ_LEN = 64   # context window

# Convert raw text → integer token IDs
vectorizer = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=SEQ_LEN + 1)
vectorizer.adapt(texts)
vocab = vectorizer.get_vocabulary()

print(f"Vocabulary: {len(vocab):,} tokens")
print(f"Sample:     {vocab[:15]}")
print(f"\n'the food was great' → {vectorizer(['the food was great']).numpy()[0][:5]}")

In [ ]:
#@title Build training sequences (input → next token) { display-mode: "form" }
all_tokens = vectorizer(np.array(texts)).numpy()
all_tokens = all_tokens[np.sum(all_tokens > 0, axis=1) > 20]  # drop very short reviews

# Input = tokens[:-1], Target = tokens shifted by one position
# This is how every LLM is trained: predict the next token
x_train = all_tokens[:, :-1]
y_train = all_tokens[:, 1:]

print(f"Training sequences: {x_train.shape[0]:,}")

---
# 3 Build the Model

#### The model uses token embeddings, position embeddings, causal self-attention, feed-forward layers, residual connections, and layer normalization.


In [ ]:
#@title Model settings { display-mode: "form" }
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 256


In [ ]:
# Build the model manually, without a TransformerBlock class

inputs = layers.Input(shape=(SEQ_LEN,), name="tokens")

# 1. Token embeddings: one vector per token ID
token_embeddings = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    name="token_embedding"
)(inputs)

# 2. Position embeddings: one vector per location in the context window
positions = tf.range(start=0, limit=SEQ_LEN, delta=1)
position_embeddings = layers.Embedding(
    input_dim=SEQ_LEN,
    output_dim=EMBED_DIM,
    name="position_embedding"
)(positions)

x = token_embeddings + position_embeddings

# 3. Causal self-attention block 1
attn_1 = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBED_DIM // NUM_HEADS,
    name="causal_attention_1"
)(x, x, use_causal_mask=True)
x = layers.LayerNormalization(name="norm_1")(x + attn_1)

ff_1 = layers.Dense(FF_DIM, activation="relu", name="ff_1_dense_1")(x)
ff_1 = layers.Dense(EMBED_DIM, name="ff_1_dense_2")(ff_1)
x = layers.LayerNormalization(name="norm_2")(x + ff_1)

# 4. Causal self-attention block 2
attn_2 = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBED_DIM // NUM_HEADS,
    name="causal_attention_2"
)(x, x, use_causal_mask=True)
x = layers.LayerNormalization(name="norm_3")(x + attn_2)

ff_2 = layers.Dense(FF_DIM, activation="relu", name="ff_2_dense_1")(x)
ff_2 = layers.Dense(EMBED_DIM, name="ff_2_dense_2")(ff_2)
x = layers.LayerNormalization(name="norm_4")(x + ff_2)

# 5. Next-token probabilities
outputs = layers.Dense(VOCAB_SIZE, activation="softmax", name="next_token_probs")(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"Parameters: {model.count_params():,}")


In [ ]:
model.summary()


---
# 4 Generation Helpers

#### The widget shows two things for any prompt: the immediate next-token distribution and a 20-token generated continuation.


In [ ]:
#@title Define generation utilities + interactive widget { display-mode: "form" }
id_to_word = dict(enumerate(vocab))


def _prompt_tokens(prompt):
    tokens = vectorizer([prompt]).numpy()[0]
    nonzero = np.where(tokens > 0)[0]
    if len(nonzero) == 0:
        return []
    return list(tokens[:nonzero[-1] + 1])


def next_token_probs(model, prompt):
    out = _prompt_tokens(prompt)
    if not out:
        return None

    padded = np.zeros(SEQ_LEN, dtype="int32")
    context = out[-SEQ_LEN:]
    padded[:len(context)] = context
    pred_pos = len(context) - 1

    return model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]


def plot_next_token_distribution(model, prompt, top_k=10):
    probs = next_token_probs(model, prompt)
    if probs is None:
        print("Type a prompt first.")
        return

    top_ids = np.argsort(probs)[-top_k:][::-1]
    words = [id_to_word[i] for i in top_ids]
    values = [probs[i] for i in top_ids]

    plt.figure(figsize=(6, 2.4))
    plt.bar(words, values)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Probability")
    plt.title("Next-token distribution")
    plt.tight_layout()
    plt.show()


def generate_text(model, prompt, length=20, temperature=0.8):
    out = _prompt_tokens(prompt)
    if not out:
        return "(type a prompt first)"

    prompt_len = len(out)

    for _ in range(length):
        padded = np.zeros(SEQ_LEN, dtype="int32")
        context = out[-SEQ_LEN:]
        padded[:len(context)] = context
        pred_pos = len(context) - 1

        probs = model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]
        probs = np.exp(np.log(probs + 1e-10) / temperature)
        probs = probs / probs.sum()

        next_id = np.random.choice(len(probs), p=probs)
        if next_id == 0:
            break
        out.append(next_id)

    prompt_words = [id_to_word[t] for t in out[:prompt_len]]
    generated_words = [id_to_word[t] for t in out[prompt_len:]]
    return f"<b>{' '.join(prompt_words)}</b> {' '.join(generated_words)}"


def make_generator_widget(model, title="Try it yourself"):
    prompt_box = widgets.Text(
        value="the food was",
        placeholder="Type a prompt...",
        description="Prompt:",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "60px"}
    )

    temperature = widgets.FloatSlider(
        value=0.8,
        min=0.2,
        max=1.5,
        step=0.1,
        description="Temp:",
        readout_format=".1f",
        layout=widgets.Layout(width="300px"),
        style={"description_width": "45px"}
    )

    button = widgets.Button(description="Generate", button_style="primary")

    chart_output = widgets.Output()
    text_output = widgets.Output(layout=widgets.Layout(min_height="50px", padding="8px"))

    def run_generation(_=None):
        with chart_output:
            chart_output.clear_output(wait=True)
            plot_next_token_distribution(model, prompt_box.value, top_k=10)

        with text_output:
            text_output.clear_output(wait=True)
            generated = generate_text(
                model,
                prompt_box.value,
                length=20,
                temperature=temperature.value
            )
            display(HTML(f"<div style='font-size:15px'>{generated}</div>"))

    button.on_click(run_generation)

    try:
        prompt_box.on_submit(run_generation)
    except Exception:
        pass

    display(HTML(f"<h4>{title}</h4>"))
    display(widgets.HBox([prompt_box, button]))
    display(temperature)
    display(HTML("<b>Immediate next-token distribution</b>"))
    display(chart_output)
    display(HTML("<b>Generated continuation: next 20 tokens</b>"))
    display(text_output)

    run_generation()


---
# 5 Phase 1 — 1 Epoch

In [ ]:
h1 = model.fit(x_train, y_train, batch_size=128, epochs=1, validation_split=0.05)

all_loss = list(h1.history['loss'])
all_val  = list(h1.history['val_loss'])
all_acc  = list(h1.history['accuracy'])

In [ ]:
make_generator_widget(model, "Phase 1 — Generate after 1 epoch")

---
# 6 Phase 2 — 5 Epochs Total

In [ ]:
h2 = model.fit(x_train, y_train, batch_size=128, epochs=4, validation_split=0.05)

all_loss += h2.history['loss']
all_val  += h2.history['val_loss']
all_acc  += h2.history['accuracy']

In [ ]:
make_generator_widget(model, "Phase 2 — Generate after 5 epochs")

---
# 7 Phase 3 — 10 Epochs Total

In [ ]:
h3 = model.fit(x_train, y_train, batch_size=128, epochs=5, validation_split=0.05)

all_loss += h3.history['loss']
all_val  += h3.history['val_loss']
all_acc  += h3.history['accuracy']

In [ ]:
make_generator_widget(model, "Phase 3 — Generate after 10 epochs")

---
# 8 Training Progress

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, len(all_loss) + 1)

ax1.plot(epochs, all_loss, 'b-o', ms=3, label='Train')
ax1.plot(epochs, all_val, 'r-o', ms=3, label='Val')
ax1.axvline(1, color='gray', ls='--', alpha=.5)
ax1.axvline(5, color='gray', ls='--', alpha=.5)
ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss (lower = better predictions)')
ax1.legend()

ax2.plot(epochs, all_acc, 'b-o', ms=3)
ax2.axvline(1, color='gray', ls='--', alpha=.5)
ax2.axvline(5, color='gray', ls='--', alpha=.5)
ax2.set(xlabel='Epoch', ylabel='Accuracy', title='Next-Token Prediction Accuracy')

plt.tight_layout()
plt.show()